# 📊 EDA — Saham TLKM.JK

Notebook ini berisi eksplorasi data historis saham Telkom Indonesia (TLKM.JK).

**Alur:**
1. Load Data
2. Assessing Data
3. Cleaning Data
4. Exploratory Analysis & Visualisasi
5. Export Data Bersih


## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 100


## 2. Load Data

Data diload dari file CSV hasil download Yahoo Finance untuk saham **TLKM.JK**.  
Range data penuh: **2011 – 2024** (akan difilter di tahap cleaning).


In [ ]:
# Load data dari folder raw
telkom_data = pd.read_csv('../data/raw/data_tlkm.csv', index_col=0)
print(f'Shape: {telkom_data.shape}')
telkom_data.head(10)


## 3. Assessing Data

Periksa struktur, tipe data, missing values, duplikat, dan nilai anomali.


In [ ]:
# Informasi umum dataframe
telkom_data.info()


In [ ]:
# Statistik deskriptif
telkom_data.describe()


In [ ]:
# Cek missing values dan duplikasi
print('=== Missing Values ===')
print(telkom_data.isnull().sum())
print()
print('=== Duplikasi ===')
print(f'Jumlah baris duplikat: {telkom_data.duplicated().sum()}')


In [ ]:
# Cek tipe data
print('=== Tipe Data ===')
print(telkom_data.dtypes)


In [ ]:
# Cek nilai 0 pada semua kolom numerik sekaligus
num_cols = ['open', 'high', 'low', 'close', 'volume']
zero_counts = (telkom_data[num_cols] == 0).sum()
print('=== Nilai 0 per Kolom ===')
print(zero_counts)
print()
print('Baris dengan volume = 0:')
print(telkom_data[telkom_data['volume'] == 0][['open', 'high', 'low', 'close', 'volume']].head())


## 4. Cleaning Data

Langkah cleaning:
- Ubah kolom `date` ke tipe `datetime`
- Handle nilai `volume = 0` dengan **interpolasi linear**  
  (alasan: harga OHLC tetap valid, volume 0 = kemungkinan data entry issue hari libur)
- Filter rentang tahun yang relevan untuk training model


In [ ]:
# Ubah kolom date ke datetime
telkom_data['date'] = pd.to_datetime(telkom_data['date'])
print(f'Tipe date setelah konversi: {telkom_data["date"].dtype}')


In [ ]:
# Handle volume = 0 → ganti ke NaN lalu interpolasi linear
telkom_data['volume'] = telkom_data['volume'].replace(0, np.nan)
telkom_data['volume'] = telkom_data['volume'].interpolate(method='linear')

# Verifikasi
print(f'Volume = 0 setelah interpolasi : {(telkom_data["volume"] == 0).sum()}')
print(f'Volume NaN setelah interpolasi  : {telkom_data["volume"].isna().sum()}')


In [ ]:
# Set date sebagai index
telkom_data.set_index('date', inplace=True)
print(f'Shape setelah set index: {telkom_data.shape}')


In [ ]:
# Filter rentang tahun
# ─────────────────────────────────────────────────────────────────
# Alasan filter ke 2023-2024:
# Data 2011-2022 mencakup banyak regime market yang berbeda
# (pre-COVID, crash COVID 2020, recovery). Untuk prediksi jangka
# pendek dengan LSTM, kita fokus ke data terbaru agar pola lebih
# representatif terhadap kondisi market saat ini.
# ─────────────────────────────────────────────────────────────────
TAHUN_MULAI = 2023
TAHUN_AKHIR = 2024

telkom_data = telkom_data[
    (telkom_data.index.year >= TAHUN_MULAI) &
    (telkom_data.index.year <= TAHUN_AKHIR)
]

print(f'Shape setelah filter   : {telkom_data.shape}')
print(f'Rentang tanggal        : {telkom_data.index.min().date()} s/d {telkom_data.index.max().date()}')


In [ ]:
# Preview data bersih
telkom_data.head()


## 5. Exploratory Analysis & Visualisasi

### 5.1 Harga Penutupan (Close) + Moving Average

In [ ]:
# Hitung Moving Average
telkom_data['MA_20'] = telkom_data['close'].rolling(window=20).mean()
telkom_data['MA_50'] = telkom_data['close'].rolling(window=50).mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(telkom_data.index, telkom_data['close'], label='Close', linewidth=1.5, color='#1f77b4')
ax.plot(telkom_data.index, telkom_data['MA_20'], label='MA-20', linestyle='--', linewidth=1.2, color='#ff7f0e')
ax.plot(telkom_data.index, telkom_data['MA_50'], label='MA-50', linestyle='--', linewidth=1.2, color='#d62728')
ax.set_title('Harga Penutupan TLKM.JK + Moving Average', fontsize=14)
ax.set_xlabel('Tanggal')
ax.set_ylabel('Harga (IDR)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 5.2 Volume Trading

In [ ]:
fig, ax = plt.subplots(figsize=(16, 3))
ax.bar(telkom_data.index, telkom_data['volume'], color='steelblue', alpha=0.6, width=1)
ax.set_title('Volume Trading Harian TLKM.JK', fontsize=14)
ax.set_xlabel('Tanggal')
ax.set_ylabel('Volume')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 5.3 Distribusi Harga OHLC

In [ ]:
price_cols = ['open', 'high', 'low', 'close']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, col in zip(axes.flatten(), price_cols):
    ax.hist(telkom_data[col], bins=30, edgecolor='black', color='steelblue', alpha=0.7)
    ax.axvline(telkom_data[col].mean(), color='red', linestyle='--',
               label=f'Mean: {telkom_data[col].mean():.0f}')
    ax.set_title(f'Distribusi {col.capitalize()}', fontsize=12)
    ax.set_xlabel('Harga (IDR)')
    ax.set_ylabel('Frekuensi')
    ax.legend(fontsize=9)

plt.suptitle('Distribusi Harga OHLC — TLKM.JK', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('=== Skewness ===')
for col in price_cols:
    print(f'{col:8s}: {telkom_data[col].skew():.4f}')


### 5.4 Distribusi Volume

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(telkom_data['volume'], bins=40, edgecolor='black', color='darkorange', alpha=0.7)
axes[0].set_title('Distribusi Volume (Raw)')
axes[0].set_xlabel('Volume')
axes[0].set_ylabel('Frekuensi')

axes[1].hist(np.log1p(telkom_data['volume']), bins=40, edgecolor='black', color='darkorange', alpha=0.7)
axes[1].set_title('Distribusi Volume (Log Scale)')
axes[1].set_xlabel('Log(Volume + 1)')
axes[1].set_ylabel('Frekuensi')

plt.suptitle('Distribusi Volume Trading', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Skewness volume (raw): {telkom_data["volume"].skew():.4f}')


### 5.5 Heatmap Korelasi Antar Fitur

In [ ]:
corr_cols = ['open', 'high', 'low', 'close', 'volume']
corr_matrix = telkom_data[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Heatmap Korelasi Fitur TLKM.JK', fontsize=13)
plt.tight_layout()
plt.show()


### 5.6 Volatilitas Harian (Daily Return)

In [ ]:
# Daily return = perubahan harga close dalam persen
telkom_data['daily_return'] = telkom_data['close'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Time series return
axes[0].plot(telkom_data.index, telkom_data['daily_return'], linewidth=0.8, color='mediumseagreen')
axes[0].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[0].set_title('Daily Return (%)')
axes[0].set_xlabel('Tanggal')
axes[0].set_ylabel('Return (%)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

# Distribusi return
axes[1].hist(telkom_data['daily_return'].dropna(), bins=40, edgecolor='black', color='mediumseagreen', alpha=0.7)
axes[1].set_title('Distribusi Daily Return (%)')
axes[1].set_xlabel('Return (%)')
axes[1].set_ylabel('Frekuensi')

plt.suptitle('Analisis Volatilitas Harian TLKM.JK', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Rata-rata daily return : {telkom_data["daily_return"].mean():.4f}%')
print(f'Std daily return       : {telkom_data["daily_return"].std():.4f}%')
print(f'Skewness               : {telkom_data["daily_return"].skew():.4f}')


## 6. Export Data Bersih

Simpan data yang sudah di-clean ke folder `processed/` untuk dipakai di tahap preprocessing & feature engineering.


In [ ]:
# Export — hanya kolom OHLCV
# MA dan daily_return akan dihitung ulang di preprocessing.py
export_cols = ['open', 'high', 'low', 'close', 'volume']
export_df = telkom_data[export_cols].copy()

output_path = '../data/processed/TLKM_cleaned.csv'
export_df.to_csv(output_path)

print(f'Data berhasil disimpan ke: {output_path}')
print(f'Shape  : {export_df.shape}')
print(f'Rentang: {export_df.index.min().date()} s/d {export_df.index.max().date()}')
